# Kaggle Titanic Dataset (Random Forest)

In [1]:
# 1) Imports and Paths
import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, RandomizedSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn import metrics
import joblib

# Paths and constants
DATA_DIR = "/home/atul-kumar/workspace/kaggle/titanic/data"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
SUBMISSION_PATH = os.path.join(DATA_DIR, "submission-rf.csv")
MODEL_DIR = os.path.join(DATA_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Paths set:", TRAIN_PATH, TEST_PATH)

Paths set: /home/atul-kumar/workspace/kaggle/titanic/data/train.csv /home/atul-kumar/workspace/kaggle/titanic/data/test.csv


In [2]:
# 2) Load Data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("Train shape:", train_df.shape, " Test shape:", test_df.shape)
train_df.head(3)

Train shape: (891, 12)  Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [3]:
# 3) Feature Engineering Reuse

def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.")
    out["FamilySize"] = out.get("SibSp", 0) + out.get("Parch", 0) + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    if "Ticket" in out.columns:
        counts = out["Ticket"].value_counts()
        out["TicketGroup"] = out["Ticket"].map(counts)
    else:
        out["TicketGroup"] = 1
    return out

train_df_fe = add_engineered_features(train_df)
test_df_fe = add_engineered_features(test_df)
print("Engineered columns present:", {c for c in train_df_fe.columns if c in ["Title","FamilySize","IsAlone","TicketGroup"]})

Engineered columns present: {'Title', 'IsAlone', 'FamilySize', 'TicketGroup'}


In [4]:
# 4) Define Preprocessing
base_features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
engineered = ["Title", "FamilySize", "IsAlone", "TicketGroup"]
all_features = base_features + engineered

numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize", "TicketGroup"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

X = train_df_fe[all_features]
y = train_df_fe["Survived"]
print("Feature matrix shape:", X.shape)

Feature matrix shape: (891, 11)


In [5]:
# 5) Random Forest Model Definition
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight=None,
)

model = Pipeline(steps=[
    ("pre", preprocess),
    ("clf", rf),
])
print(model)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'SibSp', 'Parch',
                                                   'Fare', 'FamilySize',
                                                   'TicketGroup']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_outpu

In [6]:
# 6) Cross-Validation Metrics (Accuracy and ROC-AUC)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
acc_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
auc_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"CV Accuracy: {acc_scores.mean():.4f} +/- {acc_scores.std():.4f}")
print(f"CV ROC-AUC: {auc_scores.mean():.4f} +/- {auc_scores.std():.4f}")

# Note: AUC = ∫_0^1 TPR(FPR^{-1}(x)) dx
print("AUC is the area under ROC curve: ∫_0^1 TPR(FPR^{-1}(x)) dx")

CV Accuracy: 0.8025 +/- 0.0136
CV ROC-AUC: 0.8712 +/- 0.0285
AUC is the area under ROC curve: ∫_0^1 TPR(FPR^{-1}(x)) dx


In [7]:
# 7) Hyperparameter Search with RandomizedSearchCV
param_distributions = {
    'clf__n_estimators': [300, 600, 900],
    'clf__max_depth': [None, 4, 6, 8, 12],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features': ['sqrt', 0.6, 0.8],
    'clf__bootstrap': [True, False],
}

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=25,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
    verbose=1,
)

search.fit(X, y)
print("Best params:", search.best_params_)
print("Best ROC-AUC:", round(search.best_score_, 4))

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best params: {'clf__n_estimators': 900, 'clf__min_samples_split': 5, 'clf__min_samples_leaf': 1, 'clf__max_features': 0.6, 'clf__max_depth': 12, 'clf__bootstrap': True}
Best ROC-AUC: 0.8834


In [8]:
# 8) Refit Best Model on Full Training Data
best_estimator = search.best_estimator_
best_estimator.fit(X, y)
final_model = best_estimator
print("Best model refit on full data.")

Best model refit on full data.


In [9]:
# 9) Probability Calibration (Optional)
calibrated_model = CalibratedClassifierCV(estimator=final_model, method='sigmoid', cv=5)
# Compare CV log loss between final_model and calibrated_model
logloss_cv_final = -cross_val_score(final_model, X, y, cv=cv, scoring='neg_log_loss', n_jobs=-1)
logloss_cv_calibrated = -cross_val_score(calibrated_model, X, y, cv=cv, scoring='neg_log_loss', n_jobs=-1)
print(f"LogLoss final: {logloss_cv_final.mean():.4f} +/- {logloss_cv_final.std():.4f}")
print(f"LogLoss calibrated: {logloss_cv_calibrated.mean():.4f} +/- {logloss_cv_calibrated.std():.4f}")

chosen_model = calibrated_model if logloss_cv_calibrated.mean() < logloss_cv_final.mean() else final_model
print("Chosen model:", "calibrated" if chosen_model is calibrated_model else "final (uncalibrated)")

LogLoss final: 0.4087 +/- 0.0394
LogLoss calibrated: 0.3969 +/- 0.0216
Chosen model: calibrated


In [10]:
# 10) Threshold Tuning on Validation Predictions
# Get out-of-fold probabilities for chosen_model
p_oof = cross_val_predict(chosen_model, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]

best_threshold = 0.5
best_f1 = -1
for t in np.linspace(0.2, 0.8, 121):
    preds = (p_oof >= t).astype(int)
    f1 = metrics.f1_score(y, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = float(t)

acc = metrics.accuracy_score(y, (p_oof >= best_threshold).astype(int))
print(f"Best threshold: {best_threshold:.3f} with F1={best_f1:.4f}, Accuracy={acc:.4f}")

Best threshold: 0.490 with F1=0.7908, Accuracy=0.8474


In [11]:
# 11) Permutation Feature Importance (per original input features)
from sklearn.model_selection import train_test_split

# Holdout split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Use the chosen_model directly so permutation operates on raw input columns
chosen_model.fit(X_train, y_train)

r = permutation_importance(
    chosen_model, X_valid, y_valid,
    scoring='roc_auc', n_repeats=10,
    random_state=RANDOM_STATE, n_jobs=-1
)

feature_names = list(all_features)
importances = pd.Series(r.importances_mean, index=feature_names).sort_values(ascending=False)
print("Top 15 features by permutation importance (input level):")
importances.head(15)

Top 15 features by permutation importance (input level):


Sex            0.067905
Title          0.057668
Pclass         0.034585
Fare           0.017971
Age            0.010725
FamilySize     0.007457
TicketGroup    0.004269
SibSp          0.000988
IsAlone        0.000000
Parch         -0.000975
Embarked      -0.003636
dtype: float64

In [12]:
# 12) Predict Test Set and Save Submission
X_test = test_df_fe[all_features]
probs_test = chosen_model.predict_proba(X_test)[:, 1]
labels_test = (probs_test >= best_threshold).astype(int)

submission = pd.DataFrame({
    'PassengerId': test_df_fe['PassengerId'],
    'Survived': labels_test
})
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved submission to:", SUBMISSION_PATH)
submission.head(10)

Saved submission to: /home/atul-kumar/workspace/kaggle/titanic/data/submission-rf.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


In [13]:
# 13) Save Trained Model Artifact
metadata = {
    'best_params': getattr(search, 'best_params_', None),
    'best_threshold': best_threshold,
    'cv_accuracy': float(np.mean(acc_scores)),
    'cv_auc': float(np.mean(auc_scores)),
}
artifact_path = os.path.join(MODEL_DIR, 'random-forest-titanic.joblib')
joblib.dump({'model': chosen_model, 'metadata': metadata}, artifact_path)
print("Saved model artifact to:", artifact_path)
metadata

Saved model artifact to: /home/atul-kumar/workspace/kaggle/titanic/data/models/random-forest-titanic.joblib


{'best_params': {'clf__n_estimators': 900,
  'clf__min_samples_split': 5,
  'clf__min_samples_leaf': 1,
  'clf__max_features': 0.6,
  'clf__max_depth': 12,
  'clf__bootstrap': True},
 'best_threshold': 0.49000000000000005,
 'cv_accuracy': 0.8024606113866047,
 'cv_auc': 0.8711836287983663}

In [14]:
# 14) Reproducibility: Random Seeds and Determinism
# Already set RANDOM_STATE and numpy seed. For extra determinism in terminal runs:
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
os.environ.setdefault('OMP_NUM_THREADS', '1')
print("Seeds and environment set for reproducibility.")

Seeds and environment set for reproducibility.
